# 0 · Basics

**The project.** We have satellite pictures and, for each, a second picture that says what is on the ground at every pixel: water, trees, crops, buildings and so on. We want a model that draws the second picture from the first.

GeoSave opens satellite files as standard Python data objects, ready for PyTorch and Lightning. We will follow one workflow:

Open images → inspect them → prepare tensors → train with Lightning → inspect predictions.

This is ordinary image segmentation. The data is laid out like most segmentation datasets:

```text
data/dynamic_world/
├── train/  images/<id>.tif   labels/<id>.tif
├── val/    ...
└── test/   ...
```

It holds 300 tiles of [Dynamic World](https://doi.org/10.1594/PANGAEA.933475) expert labels, each paired with the Sentinel-2 picture it was drawn on, and is published on the Hugging Face Hub as [fatmur/dynamicworld-300](https://huggingface.co/datasets/fatmur/dynamicworld-300).

Download it into `data/dynamic_world`, next to this notebook. It is about 400 MB; running the cell again only checks for changes.

In [1]:
from pathlib import Path

from huggingface_hub import snapshot_download

root = Path(snapshot_download("fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"))

Fetching 603 files:   0%|          | 0/603 [00:00<?, ?it/s]

Open one picture and its labels. `gs.open_raster` returns an ordinary `xarray.Dataset`, with one named variable per band. GeoSave adds `.gs` methods for plotting and tensor conversion.

A photo has three bands: red, green and blue. This picture has four: blue, green, red, and near-infrared, light the eye cannot see but plants reflect strongly.

GeoSave reads the RGB band choices and the label's class names and colours from the files, so we can plot them together without choosing colours ourselves.

In [ ]:
from IPython.display import display

import geosave_engine as gs

name = sorted((root / "train/images").iterdir())[0].name
image = gs.open_raster(root / "train/images" / name)
label = gs.open_raster(root / "train/labels" / name)

display(image)
gs.stack({"image": image, "label": label}).gs.plot(cols=2)

PyTorch wants tensors. GeoSave's `.gs.to_tensor()` stacks the named bands in dataset order into a tensor shaped `(bands, height, width)`, ready for a plain PyTorch `Dataset` and `DataLoader`.

Two details of this data:

- Each picture carries its date as an axis of length one; `.isel(time=0)` takes that one date.
- The pixels are stored as whole numbers, 10,000 times the share of light reflected, so dividing by 10,000 plays the part `/ 255` plays for a photo.

In [3]:
import torch
from torch.utils.data import DataLoader, Dataset


class Tiles(Dataset):
    def __init__(self, split):
        self.folder = root / split
        self.names = sorted(path.name for path in (self.folder / "images").iterdir())

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = gs.open_raster(self.folder / "images" / name).isel(time=0)
        label = gs.open_raster(self.folder / "labels" / name).isel(time=0)
        return image.gs.to_tensor(dtype="float32") / 10_000, label.gs.to_tensor(dtype="int64")[0]


train = Tiles("train")
val = Tiles("val")
train_loader = DataLoader(train, batch_size=8, shuffle=True)
val_loader = DataLoader(val, batch_size=8)

image, label = train[0]
image.shape, label.shape

(torch.Size([4, 510, 510]), torch.Size([510, 510]))

The model has the three parts most segmentation models have: an encoder that turns pixels into features, a decoder that brings the features back to full size, and a head that scores each of the 10 classes. The encoder halves the height and width and the decoder doubles them, so this tiny model expects even tile dimensions, as this dataset has. Printing the model shows its layers.

In [4]:
from torch import nn


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(4, 16, 3, stride=2, padding=1), nn.ReLU())

    def forward(self, image):
        return self.net(image)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(16, 16, 3, padding=1), nn.ReLU())

    def forward(self, features):
        return self.net(features)


class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Conv2d(16, 10, 1)

    def forward(self, features):
        return self.net(features)


model = nn.Sequential(Encoder(), Decoder(), Head())
model

Sequential(
  (0): Encoder(
    (net): Sequential(
      (0): Conv2d(4, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ReLU()
    )
  )
  (1): Decoder(
    (net): Sequential(
      (0): Upsample(scale_factor=2.0, mode='nearest')
      (1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (2): ReLU()
    )
  )
  (2): Head(
    (net): Conv2d(16, 10, kernel_size=(1, 1), stride=(1, 1))
  )
)

Training is plain Lightning. First a module that says how a batch is scored. Label 255 marks pixels a labeller left unmarked, so the loss and the metrics ignore it, as segmentation code usually does. The metrics come from [torchmetrics](https://lightning.ai/docs/torchmetrics): mean intersection over union (mIoU), the usual segmentation score, and pixel accuracy.

Given an example input, Lightning's `ModelSummary` lists every layer with the size of what goes in and comes out.

In [5]:
import lightning as L
from lightning.pytorch.utilities.model_summary.model_summary import ModelSummary
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.example_input_array = torch.zeros(1, 4, 510, 510)
        metrics = MetricCollection({
            "miou": MulticlassJaccardIndex(num_classes=10, ignore_index=255),
            "accuracy": MulticlassAccuracy(num_classes=10, ignore_index=255, average="micro"),
        })
        self.val_metrics = metrics.clone(prefix="val_")

    def forward(self, image):
        return self.model(image)

    def training_step(self, batch, batch_index):
        image, label = batch
        loss = nn.functional.cross_entropy(self(image), label, ignore_index=255)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch
        self.val_metrics(self(image), label)
        self.log_dict(self.val_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(model)
ModelSummary(module, max_depth=-1)

   | Name                 | Type                   | Params | Mode  | FLOPs  | In sizes          | Out sizes        
--------------------------------------------------------------------------------------------------------------------------
0  | model                | Sequential             | 3.1 K  | train | 1.4 B  | [1, 4, 510, 510]  | [1, 10, 510, 510]
1  | model.0              | Encoder                | 592    | train | 74.9 M | [1, 4, 510, 510]  | [1, 16, 255, 255]
2  | model.0.net          | Sequential             | 592    | train | 74.9 M | [1, 4, 510, 510]  | [1, 16, 255, 255]
3  | model.0.net.0        | Conv2d                 | 592    | train | 74.9 M | [1, 4, 510, 510]  | [1, 16, 255, 255]
4  | model.0.net.1        | ReLU                   | 0      | train | 0      | [1, 16, 255, 255] | [1, 16, 255, 255]
5  | model.1              | Decoder                | 2.3 K  | train | 1.2 B  | [1, 16, 255, 255] | [1, 16, 510, 510]
6  | model.1.net          | Sequential             | 2.3 K

Then a Lightning `Trainer` runs one epoch with five training batches and two validation batches. These integer limits count batches; the loaders stay unchanged. `log_every_n_steps=1` records the loss at every step, even in this short run.

The cell displays the training loss and validation scores. Remove the batch limits and increase `max_epochs` when you want to train for longer.

In [ ]:
trainer = L.Trainer(
    max_epochs=1,
    limit_train_batches=5,
    limit_val_batches=2,
    log_every_n_steps=1,
)
trainer.fit(module, train_loader, val_loader)
trainer.callback_metrics

Lightning logs to TensorBoard by default, into `lightning_logs/`. Open it here to see the loss and the validation metrics; each new run adds a version you can compare.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

Predict a validation tile: the model scores every class at every pixel, and `argmax` keeps the best. Copy those numbers into the label's xarray band, keeping its coordinates, class names and colours. GeoSave can then plot the label beside the prediction with the same legend.

In [ ]:
image, _ = val[0]
module.eval()
with torch.no_grad():
    prediction = module(image[None])[0].argmax(0)

labels = gs.open_raster(root / "val/labels" / val.names[0]).isel(time=0)

predicted = labels["label"].copy(
    data=prediction.cpu().numpy().astype("uint8")
).to_dataset()

gs.stack({"label": labels, "prediction": predicted}).gs.plot(cols=2)

## What happened

We opened satellite files as xarray Datasets, used GeoSave to plot them and turn their bands into tensors, trained with plain PyTorch and Lightning, and put a prediction back on the tile's grid for comparison.

The displayed metrics and TensorBoard curves show what this run measured. Five training batches demonstrate the loop, but are too few to judge model quality. Expect poor predictions; train for longer before comparing models.

The rest of the series improves this same project one step at a time:

| Next | Question |
| --- | --- |
| 1 · Model | How do I grow this model, swap its parts and compare runs? |
| 2 · Inputs | What do the four numbers mean, and how do I give the model better inputs? |